# AutoClips on Google Colab (GPU)

Runs the full API + Telegram bot with **CUDA Whisper** (5-10x faster transcription than CPU).

Steps once per browser:
1. **Runtime â–¸ Change runtime type â–¸ T4 GPU** (top-right).
2. Run cells top-to-bottom.
3. Keep this tab open while working â€” Colab kills the session when idle (~90 min) or after ~12h.

Stores `storage/` (downloads, transcripts, clips) on Google Drive so nothing is lost between sessions.

> **Updated:** bounded H.264 encode (`RENDER_CRF`), Telegram 50 MB auto-recompress, 60 s viral-clip clamp, readable captions + accurate seeks. The `.env` cell below includes all new knobs â€” re-run it once to pick them up.


## 1. Mount Google Drive (persists downloads/clips/transcripts)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
print("Drive mounted âœ“")

## 2. Upload the project code

Choose **one**:
- **(a)** Zip the `autoclips-python` folder locally and upload it via the file browser on the left, or
- **(b)** `git clone` a remote copy (uncomment the cell).

In [ ]:
import os, shutil, zipfile

WORK = "/content/autoclips-python"
os.makedirs(WORK, exist_ok=True)

# (b) unchanged? uncomment and use the subprocess cell below instead.
print("Use the file browser to upload your zip inside /content/, then run the next cell.")

In [ ]:
import glob, os, shutil, zipfile

zips = sorted(glob.glob("/content/*.zip"))
if not zips:
    raise SystemExit("No .zip found in /content/. Upload your autoclips-python.zip first.")
z = zips[-1]
print("Extracting", z)
with zipfile.ZipFile(z) as zf:
    zf.extractall("/content/")
os.remove(z)

# If the zip has one top-level folder that isn't "autoclips-python", rename it.
folders = [d for d in os.listdir("/content")
           if os.path.isdir(f"/content/{d}") and not d.startswith(".")]
if "autoclips-python" not in folders and len(folders) == 1:
    os.rename(f"/content/{folders[0]}", WORK)
print("Project ready at", WORK)
print(sorted(os.listdir(WORK))[:15])

## 3. Install ffmpeg + Python deps

In [ ]:
!apt-get -qq update && apt-get -qq install -y ffmpeg fonts-dejavu fonts-noto-core libgl1 libglib2.0-0 nodejs > /dev/null
print("ffmpeg:", !ffmpeg -version | head -1)
# yt-dlp needs a JS runtime for YouTube DASH formats (else ".part rename" failures).
# nodejs satisfies it -- app code passes --js-runtimes node automatically.
!node --version 2>&1 | head -1
!which deno node nodejs 2>/dev/null || echo "NO js runtime found -- downloads may fail; re-run this cell"


In [ ]:
%cd /content/autoclips-python
# NOTE: Colab already ships CUDA torch â€” pip reuses it (do NOT force a torch reinstall).
# opencv-python needs libGL (installed in the cell above).
!pip -q install -r requirements.txt
import torch
print("deps installed. torch", torch.__version__, "| cuda_available:", torch.cuda.is_available())
try:
    import ultralytics
    print('ultralytics', ultralytics.__version__, 'OK (YOLO face detection active)')
except ImportError:
    raise SystemExit('ultralytics NOT installed — FACE_MODEL=yolo would silently fall back to YuNet on every frame. '
                     'Re-run this pip cell (check its output for errors), then re-run the server cell.')


## 4. Verify the GPU is visible to faster-whisper

In [ ]:
import ctranslate2
print("CUDA devices visible to CTranslate2:", ctranslate2.get_cuda_device_count())
if ctranslate2.get_cuda_device_count() == 0:
    raise SystemExit("No GPU! Go to Runtime > Change runtime type > T4 GPU, then restart.")

## 5. Configure `.env`

Fill in the required values (Telegram + Gemini). Kept in Drive so you don't retype each session.

Optional (fixes YouTube bot-checks): upload `cookies.txt` to `/content/drive/MyDrive/autoclips-config/` before running the next cell — it is picked up automatically as `YOUTUBE_COOKIES_FILE`.


In [ ]:
import os
import os

WORK = "/content/autoclips-python"
CONFIG_DIR = "/content/drive/MyDrive/autoclips-config"
os.makedirs(CONFIG_DIR, exist_ok=True)
os.chdir(WORK)

# â”€â”€ Edit these three values â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
TELEGRAM_BOT_TOKEN = "123456:ABC-DEF..."
TELEGRAM_CHAT_ID   = "your-chat-id"
GEMINI_API_KEY     = "your-gemini-key"
# â”€â”€ Tuned defaults (change only if you know why) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
WHISPER_MODEL_SIZE = "large-v3-turbo"  # small | medium | large-v3-turbo (T4 sweet spot)
FFMPEG_PRESET      = "veryfast"        # veryfast (sharp) | ultrafast (faster, blockier)
RENDER_CRF         = "20"              # 18-22 sharp; higher = smaller but blockier
FACE_MODEL         = "yolo"            # yolo (GPU) | yunet (no dep) | res10 (legacy)
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

DRIVE_COOKIES = os.path.join(CONFIG_DIR, "cookies.txt")
cookies = DRIVE_COOKIES if os.path.exists(DRIVE_COOKIES) else ""
if cookies:
    print("Using YouTube cookies:", cookies, "(fixes 'Sign in to confirm you're not a bot')")
else:
    print("No cookies.txt in", CONFIG_DIR, "-- YouTube bot-checks may fail; see Notes cell to add it")

DRIVE_ENV = os.path.join(CONFIG_DIR, ".env")
if os.path.exists(DRIVE_ENV) and TELEGRAM_BOT_TOKEN.startswith("123456"):
    # Returning session: reuse the Drive .env you saved last time.
    _saved = open(DRIVE_ENV).read()
    # Old Drive .env files predate YOUTUBE_COOKIES_FILE - patch it in so a
    # newly uploaded cookies.txt takes effect without retyping everything.
    if cookies and "YOUTUBE_COOKIES_FILE" not in _saved:
        _saved = _saved.rstrip("\n") + f"\nYOUTUBE_COOKIES_FILE={cookies}\n"
        open(DRIVE_ENV, "w").write(_saved)
    open(".env", "w").write(_saved)
    print("Loaded existing Drive .env âœ“ (edit " + DRIVE_ENV + " to change values)")
else:
    env = f"""TELEGRAM_BOT_TOKEN={TELEGRAM_BOT_TOKEN}
TELEGRAM_CHAT_ID={TELEGRAM_CHAT_ID}
GEMINI_API_KEY={GEMINI_API_KEY}
WHISPER_MODEL_SIZE={WHISPER_MODEL_SIZE}
WHISPER_DEVICE=cuda
WHISPER_COMPUTE_TYPE=float16
WHISPER_VAD_FILTER=true
WHISPER_MIN_SILENCE_MS=400
WHISPER_SPEECH_PAD_MS=200
FFMPEG_PRESET={FFMPEG_PRESET}
RENDER_CRF={RENDER_CRF}
RENDER_AUDIO_BITRATE=128k
RENDER_FPS=30
TELEGRAM_MAX_VIDEO_MB=50.0
MAX_CLIP_DURATION_SEC=60.0
MIN_CLIP_DURATION_SEC=15.0
FACE_MODEL={FACE_MODEL}
FACE_SAMPLE_FPS=1.0
FACE_SMOOTH_WINDOW=5
FACE_MAX_PAN_PX_PER_SEC=200.0
SMOOTH_CROP=true
SNAP_TO_SILENCE=true
SNAP_WINDOW_SEC=0.8
LOG_LEVEL=INFO
YOUTUBE_COOKIES_FILE={cookies}
"""
    open(".env", "w").write(env)
    open(DRIVE_ENV, "w").write(env)
    print("Wrote .env with CUDA + quality settings âœ“ (also saved to Drive)")


## 6. Keep storage/ on Drive (survives restarts)

Downloads, transcripts, clips and logs persist under your Drive. First run moves the folder; later runs reuse it.

In [ ]:
import os, shutil
os.chdir("/content/autoclips-python")

STORE = "/content/drive/MyDrive/autoclips-storage"
for d in ["downloads", "clips", "tmp", "videos", "logs"]:
    os.makedirs(f"{STORE}/{d}", exist_ok=True)

# Point storage/ at Drive exactly ONCE (old code re-linked inside the loop).
if os.path.islink("storage"):
    print("storage/ already linked ->", os.readlink("storage"))
else:
    shutil.rmtree("storage", ignore_errors=True)
    os.symlink(STORE, "storage")
    print("storage/ ->", os.readlink("storage"))


## 7. Start the server + Telegram bot (background)

Runs uvicorn (which also starts the bot) in the background. Watch `storage/logs/app.log` in cells 8-9.

In [ ]:
import os, subprocess, sys, time, urllib.request
os.chdir("/content/autoclips-python")
assert os.path.exists(".env"), "No .env â€” run the Configure .env cell first!"

# Kill any previous instance from an old session of this same run
subprocess.run(["pkill", "-f", "uvicorn app.main:app"], capture_output=True)
time.sleep(2)

log = open("storage/logs/colab_server.log", "a", buffering=1)
subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=log, stderr=subprocess.STDOUT,
)
print("Server starting... waiting for /health")
for i in range(30):
    try:
        urllib.request.urlopen("http://localhost:8000/health", timeout=5)
        print("Server is UP âœ“ â€” message your bot a YouTube link from Telegram.")
        break
    except Exception:
        time.sleep(5)
else:
    print("Still not up â€” check storage/logs/colab_server.log and storage/logs/app.log")


In [ ]:
# Verify the quality-fix settings the server actually loaded
import os
os.chdir("/content/autoclips-python")
from app.config import settings
print("whisper:", settings.WHISPER_MODEL_SIZE, settings.WHISPER_DEVICE, settings.WHISPER_COMPUTE_TYPE)
print("encode: preset=%s crf=%s audio=%s fps=%s" % (settings.FFMPEG_PRESET, settings.RENDER_CRF, settings.RENDER_AUDIO_BITRATE, settings.RENDER_FPS))
print("caps: telegram=%.1fMB max_clip=%.0fs min_clip=%.0fs" % (settings.TELEGRAM_MAX_VIDEO_MB, settings.MAX_CLIP_DURATION_SEC, settings.MIN_CLIP_DURATION_SEC))
print("face:", settings.FACE_MODEL, "snap:", settings.SNAP_TO_SILENCE, settings.SNAP_WINDOW_SEC)
import json, urllib.request
print(json.dumps(json.loads(urllib.request.urlopen("http://localhost:8000/health", timeout=10).read()), indent=1))


## 7b. YOLO self-test: timestamp clip straight to Telegram

Runs `scripts/test_yolo_telegram.py` — loads the YOLO face model, probes one frame for faces, renders YOUR timestamp as a vertical clip (reusing the cached download in `storage/downloads/`, or a local MP4), and sends it to Telegram. No Gemini, no polling. Edit the test values first (pick a moment where a person is on screen).


In [ ]:
# EDIT ME: video source + timestamp to test
TEST_VIDEO_URL = ''  # e.g. 'https://www.youtube.com/watch?v=VIDEO_ID' (download cached in storage/downloads/)
TEST_LOCAL_VIDEO = ''  # or e.g. '/content/my_video.mp4' (upload via file browser first, skips download)
TEST_START = '00:01:20'  # HH:MM:SS, MM:SS, or seconds
TEST_END = '00:01:45'
TEST_SUBTITLES = 'none'  # none (fastest) | english | native | hinglish

import os
os.chdir('/content/autoclips-python')
assert os.path.exists('.env'), 'No .env — run the Configure .env cell first!'
assert os.path.exists('scripts/test_yolo_telegram.py'), 'Test script missing — re-upload the repo zip!'

if TEST_LOCAL_VIDEO:
    assert os.path.exists(TEST_LOCAL_VIDEO), f'Local video not found: {TEST_LOCAL_VIDEO}'
    src_args = f'--video "{TEST_LOCAL_VIDEO}"'
else:
    assert TEST_VIDEO_URL and 'VIDEO_ID' not in TEST_VIDEO_URL, 'Set TEST_VIDEO_URL (or TEST_LOCAL_VIDEO) first!'
    src_args = f'--url "{TEST_VIDEO_URL}"'

!python scripts/test_yolo_telegram.py {src_args} --start "{TEST_START}" --end "{TEST_END}" --subtitles {TEST_SUBTITLES}


## 8. Watch logs (re-run this cell anytime)

In [ ]:
!tail -n 40 storage/logs/app.log

## 9. Keep the session alive while you work

Run once â€” a background thread polls the server every 30s so the session doesn't go idle.
Still subject to Colab's ~12h hard cap; restart when it ends.

In [ ]:
import threading, time, urllib.request

def keep_alive():
    while True:
        try:
            urllib.request.urlopen("http://localhost:8000/health", timeout=5)
        except Exception:
            pass
        time.sleep(30)

t = threading.Thread(target=keep_alive, daemon=True)
t.start()
print("Keep-alive running. When done, re-run cell 7 next session to restart the server.")

## Notes / troubleshooting
- **New quality knobs (re-run the `.env` cell once to get them):** `RENDER_CRF=20`, `RENDER_AUDIO_BITRATE=128k`, `RENDER_FPS=30`, `TELEGRAM_MAX_VIDEO_MB=50`, `MAX_CLIP_DURATION_SEC=60` (overlong AI picks auto-clamp), `MIN_CLIP_DURATION_SEC=15`.
- **Oversize clips no longer die:** renders use bounded bitrate + `faststart`, and anything still over 50 MB is auto-recompressed to 720p before the Telegram send.
- **Transcription is GPU â€” check it:** `WHISPER_DEVICE=cuda`, `WHISPER_COMPUTE_TYPE=float16`, `WHISPER_VAD_FILTER=true` in `.env`. First transcribe downloads the model (~1.6GB for `large-v3-turbo`) so it's slower once.
- **Face on T4:** `FACE_MODEL=yolo` uses the T4 GPU at 1fps; on CPU boxes set `FACE_MODEL=yunet` (no new dep) or `res10`.
- **Session lost?** Drive holds `.env` (`autoclips-config/`) and `storage/`. Re-run cells 5-7 only â€” the `.env` cell reloads your saved Drive copy.
- **Rate limits:** Telegram bot uses long polling â€” no public URL needed.
- **YouTube uploads (Approve buttons)** need `YT_*` vars added to `.env` â€” same as local.
- **Out of VRAM:** switch to `WHISPER_MODEL_SIZE=medium`, and transcribe before face detection (sequential) so both don't sit in VRAM at once.
- **YouTube 'Sign in to confirm you're not a bot' (Colab IPs get blocked):** export cookies.txt from your logged-in desktop browser (extension 'Get cookies.txt LOCALES' on youtube.com), upload it to `/content/drive/MyDrive/autoclips-config/cookies.txt`, re-run the `.env` cell + server cell. The downloader also retries android/ios player clients automatically (reduced quality) when the web client is blocked.
- **'No supported JavaScript runtime' warning:** re-run the system-deps cell (`apt-get install -y nodejs`) and re-upload the latest repo zip — old builds hardcoded `node:/tools/node/bin/node` (Kaggle path) which doesn't exist on Colab.
